<h1 style="font-size: 26px;">SQL Queries Start From Here</h1>


In [4]:
import pandas as pd
from getpass import getpass
from sqlalchemy import URL, create_engine

engine = create_engine(
    URL.create(
        "postgresql+psycopg2",
        username="postgres",
        password=getpass("PostgreSQL password: "),
        host="localhost",
        port=5432,
        database="sunrise_coffee_db",
    )
)

Which store formats generate the highest average net profit margin, and why?

Transport-Hub stores lead with a 22.53% average margin versus 14.34% for High-Street stores, earning more monthly revenue (£66,386 vs £62,140).
They also average lower rent (£5,882 vs £6,480) and staffing costs (£17,270 vs £19,369), which likely contributes to their stronger margins; this comparison suggests drivers but doesn’t prove causation.

In [2]:
query = """
SELECT
    s.store_format,
    ROUND(AVG(f.net_profit_margin_pct) * 100, 2) AS avg_margin_pct,
    ROUND(AVG(f.net_profit_gbp), 2) AS avg_monthly_profit_gbp,
    ROUND(AVG(f.total_monthly_revenue_gbp), 2) AS avg_monthly_revenue_gbp,
    ROUND(AVG(f.monthly_rent_gbp), 2) AS avg_monthly_rent_gbp,
    ROUND(AVG(f.monthly_staffing_cost_gbp), 2) AS avg_monthly_staffing_cost_gbp
FROM fact_store_monthly_profitability AS f
JOIN dim_store AS s
    ON s.store_id = f.store_id
GROUP BY s.store_format
ORDER BY avg_margin_pct DESC;
"""

results = pd.read_sql_query(query, engine)
display(results)

,store_format,avg_margin_pct,avg_monthly_profit_gbp,avg_monthly_revenue_gbp,avg_monthly_rent_gbp,avg_monthly_staffing_cost_gbp
0,Transport-Hub,22.53,16689.82,66385.96,5881.75,17269.83
1,High-Street,14.34,11550.57,62140.42,6479.94,19368.78


2......
Which stores are consistently loss-making across 2023–2024 rather than experiencing temporary losses?

In [6]:
query = """
SELECT 
    f.store_id,
    s.store_name,
    s.store_format,
    s.region,
    COUNT(*) AS total_months,
    SUM(CASE WHEN f.net_profit_gbp < 0 THEN 1 ELSE 0 END) AS loss_months,
    ROUND(
        (SUM(CASE WHEN f.net_profit_gbp < 0 THEN 1 ELSE 0 END)::NUMERIC / COUNT(*)) * 100, 
        2
    ) AS loss_month_pct,
    ROUND(SUM(f.net_profit_gbp), 2) AS total_net_profit_gbp,
    ROUND(AVG(f.net_profit_gbp), 2) AS avg_monthly_net_profit_gbp
FROM fact_store_monthly_profitability f
JOIN dim_store s ON s.store_id = f.store_id
GROUP BY f.store_id, s.store_name, s.store_format, s.region
HAVING SUM(f.net_profit_gbp) < 0
ORDER BY total_net_profit_gbp ASC;
"""

results = pd.read_sql_query(query, engine)
display(results)

,store_id,store_name,store_format,region,total_months,loss_months,loss_month_pct,total_net_profit_gbp,avg_monthly_net_profit_gbp
0,32,Sunrise Coffee South London Corner,High-Street,London,24,13,54.17,-50494.58,-2103.94


Which stores generate high revenue but comparatively low profit, and what is driving the gap?

In [7]:
query = """
SELECT 
    f.store_id,
    s.store_name,
    s.store_format,
    s.region,
    ROUND(SUM(f.total_monthly_revenue_gbp), 2) AS total_revenue_gbp,
    ROUND(SUM(f.net_profit_gbp), 2) AS total_net_profit_gbp,
    ROUND(
        (SUM(f.net_profit_gbp) / NULLIF(SUM(f.total_monthly_revenue_gbp), 0)) * 100, 
        2
    ) AS net_profit_margin_pct,
    ROUND(SUM(f.total_monthly_cogs_gbp), 2) AS total_cogs_gbp,
    ROUND(SUM(f.monthly_rent_gbp), 2) AS total_rent_gbp,
    ROUND(SUM(f.monthly_staffing_cost_gbp), 2) AS total_staffing_cost_gbp,
    ROUND(SUM(f.loyalty_redemption_cost_gbp), 2) AS total_loyalty_cost_gbp
FROM fact_store_monthly_profitability f
JOIN dim_store s ON s.store_id = f.store_id
GROUP BY f.store_id, s.store_name, s.store_format, s.region
HAVING 
    SUM(f.total_monthly_revenue_gbp) > (
        SELECT AVG(total_store_revenue) 
        FROM (
            SELECT SUM(total_monthly_revenue_gbp) AS total_store_revenue 
            FROM fact_store_monthly_profitability 
            GROUP BY store_id
        ) sub_rev
    )
ORDER BY net_profit_margin_pct ASC
LIMIT 10;
"""

results = pd.read_sql_query(query, engine)
display(results)

,store_id,store_name,store_format,region,total_revenue_gbp,total_net_profit_gbp,net_profit_margin_pct,total_cogs_gbp,total_rent_gbp,total_staffing_cost_gbp,total_loyalty_cost_gbp
0,5,Sunrise Coffee Canary Wharf Quarter,High-Street,London,1512182.70,131422.52,8.69,556310.82,298848.97,499705.94,25894.45
1,46,Sunrise Coffee East London Lounge,High-Street,London,1713156.43,164644.15,9.61,646542.96,294082.98,581024.70,26861.64
2,39,Sunrise Coffee Central London House,High-Street,London,1497966.27,201618.64,13.46,545027.19,263417.91,457961.39,29941.14
3,21,Sunrise Coffee Birmingham Roastery,High-Street,West-Midlands,1570209.70,239687.32,15.26,612084.25,87911.10,600473.39,30053.64
4,23,Sunrise Coffee Edinburgh Corner,High-Street,Scotland,1533243.80,234759.12,15.31,583846.99,82856.30,604798.07,26983.32
5,22,Sunrise Coffee Derby Lounge,High-Street,East-Midlands,1510338.38,237267.24,15.71,563112.16,140597.30,538148.64,31213.04
6,27,Sunrise Coffee Plymouth Express,High-Street,South-West,1548221.65,245754.19,15.87,581705.99,134837.11,558034.10,27890.26
7,36,Sunrise Coffee Leicester Express,High-Street,East-Midlands,1530939.77,251960.18,16.46,584790.45,210885.89,454958.31,28344.94
8,24,Sunrise Coffee Cheltenham Quarter,High-Street,South-West,1619195.19,279844.25,17.28,596301.47,222955.07,490034.14,30060.26
9,37,Sunrise Coffee Ipswich House,High-Street,East-of-England,1586068.04,274725.12,17.32,595881.86,164502.43,522631.98,28326.65


Does higher store rent translate into higher revenue or profit, or are some stores paying disproportionately high rent?

In [9]:
query = """
WITH store_metrics AS (
    SELECT
        s.store_id,
        s.store_name,
        s.store_format,
        AVG(f.monthly_rent_gbp) AS avg_monthly_rent_gbp,
        AVG(f.total_monthly_revenue_gbp) AS avg_monthly_revenue_gbp,
        AVG(f.net_profit_gbp) AS avg_monthly_profit_gbp,
        AVG(
            f.monthly_rent_gbp
            / NULLIF(f.total_monthly_revenue_gbp, 0)
        ) * 100 AS rent_share_of_revenue_pct
    FROM fact_store_monthly_profitability AS f
    JOIN dim_store AS s
        ON s.store_id = f.store_id
    WHERE f.month_id BETWEEN '2023-01' AND '2024-12'
      AND f.total_monthly_revenue_gbp > 0
    GROUP BY s.store_id, s.store_name, s.store_format
),
format_benchmarks AS (
    SELECT
        store_format,
        AVG(rent_share_of_revenue_pct) AS format_avg_rent_share_pct
    FROM store_metrics
    GROUP BY store_format
),
rent_relationships AS (
    SELECT
        CORR(avg_monthly_rent_gbp, avg_monthly_revenue_gbp) AS rent_revenue_corr,
        CORR(avg_monthly_rent_gbp, avg_monthly_profit_gbp) AS rent_profit_corr
    FROM store_metrics
)
SELECT
    m.store_id,
    m.store_name,
    m.store_format,
    ROUND(m.avg_monthly_rent_gbp, 2) AS avg_monthly_rent_gbp,
    ROUND(m.avg_monthly_revenue_gbp, 2) AS avg_monthly_revenue_gbp,
    ROUND(m.avg_monthly_profit_gbp, 2) AS avg_monthly_profit_gbp,
    ROUND(m.rent_share_of_revenue_pct::numeric, 2) AS rent_share_of_revenue_pct,
    ROUND(b.format_avg_rent_share_pct::numeric, 2) AS format_avg_rent_share_pct,
    ROUND(
        (m.rent_share_of_revenue_pct - b.format_avg_rent_share_pct)::numeric,
        2
    ) AS rent_share_above_format_avg_pct_points,
    ROUND(r.rent_revenue_corr::numeric, 3) AS rent_revenue_correlation,
    ROUND(r.rent_profit_corr::numeric, 3) AS rent_profit_correlation
FROM store_metrics AS m
JOIN format_benchmarks AS b
    ON b.store_format = m.store_format
CROSS JOIN rent_relationships AS r
ORDER BY rent_share_above_format_avg_pct_points DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_id,store_name,store_format,avg_monthly_rent_gbp,avg_monthly_revenue_gbp,avg_monthly_profit_gbp,rent_share_of_revenue_pct,format_avg_rent_share_pct,rent_share_above_format_avg_pct_points,rent_revenue_correlation,rent_profit_correlation
0,53,Sunrise Coffee Canary Wharf Express,High-Street,12262.10,56296.83,1783.64,24.21,11.37,12.84,0.147,-0.393
1,15,Sunrise Coffee West London Quarter,High-Street,14302.68,64860.00,13322.60,22.90,11.37,11.53,0.147,-0.393
2,39,Sunrise Coffee Central London House,High-Street,10975.75,62415.26,8400.78,22.19,11.37,10.82,0.147,-0.393
3,43,Sunrise Coffee North London Yard,High-Street,12292.66,60149.25,3090.78,21.06,11.37,9.69,0.147,-0.393
4,5,Sunrise Coffee Canary Wharf Quarter,High-Street,12452.04,63007.61,5475.94,20.68,11.37,9.31,0.147,-0.393
5,46,Sunrise Coffee East London Lounge,High-Street,12253.46,71381.52,6860.17,18.30,11.37,6.93,0.147,-0.393
6,29,Sunrise Coffee Dundee Station,High-Street,8958.17,59786.51,9889.80,17.83,11.37,6.46,0.147,-0.393
7,20,Sunrise Coffee Guildford Hub,High-Street,9104.74,56724.49,7459.52,16.80,11.37,5.43,0.147,-0.393
8,14,Sunrise Coffee Oxford Lounge,High-Street,9153.45,61778.72,14065.98,16.35,11.37,4.98,0.147,-0.393
9,13,Sunrise Coffee Norwich Station,High-Street,9024.95,60276.24,14279.31,16.28,11.37,4.91,0.147,-0.393


Which stores improved their profitability the most from 2023 to 2024?

In [10]:
query = """
WITH store_year_metrics AS (
    SELECT
        s.store_id,
        s.store_name,
        COUNT(DISTINCT f.month_id) FILTER (
            WHERE LEFT(f.month_id, 4) = '2023'
        ) AS months_2023,
        COUNT(DISTINCT f.month_id) FILTER (
            WHERE LEFT(f.month_id, 4) = '2024'
        ) AS months_2024,
        SUM(f.net_profit_gbp) FILTER (
            WHERE LEFT(f.month_id, 4) = '2023'
        ) AS net_profit_2023_gbp,
        SUM(f.net_profit_gbp) FILTER (
            WHERE LEFT(f.month_id, 4) = '2024'
        ) AS net_profit_2024_gbp,
        SUM(f.total_monthly_revenue_gbp) FILTER (
            WHERE LEFT(f.month_id, 4) = '2023'
        ) AS revenue_2023_gbp,
        SUM(f.total_monthly_revenue_gbp) FILTER (
            WHERE LEFT(f.month_id, 4) = '2024'
        ) AS revenue_2024_gbp
    FROM fact_store_monthly_profitability AS f
    JOIN dim_store AS s
        ON s.store_id = f.store_id
    WHERE f.month_id >= '2023-01'
      AND f.month_id < '2025-01'
    GROUP BY s.store_id, s.store_name
),
complete_years AS (
    SELECT *
    FROM store_year_metrics
    WHERE months_2023 = 12
      AND months_2024 = 12
)
SELECT
    store_id,
    store_name,
    ROUND(net_profit_2023_gbp, 2) AS net_profit_2023_gbp,
    ROUND(net_profit_2024_gbp, 2) AS net_profit_2024_gbp,
    ROUND((net_profit_2024_gbp - net_profit_2023_gbp)::numeric, 2)
        AS net_profit_change_gbp,
    ROUND(
        (100.0 * net_profit_2023_gbp / NULLIF(revenue_2023_gbp, 0))::numeric,
        2
    ) AS profit_margin_2023_pct,
    ROUND(
        (100.0 * net_profit_2024_gbp / NULLIF(revenue_2024_gbp, 0))::numeric,
        2
    ) AS profit_margin_2024_pct,
    ROUND(
        (
            100.0 * net_profit_2024_gbp / NULLIF(revenue_2024_gbp, 0)
            - 100.0 * net_profit_2023_gbp / NULLIF(revenue_2023_gbp, 0)
        )::numeric,
        2
    ) AS margin_improvement_percentage_points
FROM complete_years
ORDER BY margin_improvement_percentage_points DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_id,store_name,net_profit_2023_gbp,net_profit_2024_gbp,net_profit_change_gbp,profit_margin_2023_pct,profit_margin_2024_pct,margin_improvement_percentage_points
0,45,Sunrise Coffee Liverpool Collective,80689.73,197186.52,116496.79,13.05,25.48,12.43
1,42,Sunrise Coffee Belfast Yard,150444.03,267233.65,116789.62,23.37,33.33,9.96
2,24,Sunrise Coffee Cheltenham Quarter,94809.96,185034.29,90224.33,12.69,21.22,8.53
3,37,Sunrise Coffee Ipswich House,98354.53,176370.59,78016.06,13.07,21.16,8.10
4,48,Sunrise Coffee Reading Hub,148060.19,222801.42,74741.23,20.37,27.36,6.99
5,25,Sunrise Coffee Glasgow Express,109759.22,170854.92,61095.70,14.01,20.82,6.81
6,44,Sunrise Coffee Newcastle Collective,73872.92,129400.87,55527.95,10.62,17.24,6.62
7,4,Sunrise Coffee Chester Brew Bar,51943.34,99295.70,47352.36,7.43,12.91,5.48
8,53,Sunrise Coffee Canary Wharf Express,2241.41,40565.98,38324.57,0.35,5.78,5.43
9,15,Sunrise Coffee West London Quarter,133735.85,186006.60,52270.75,17.87,23.02,5.15


Are high-footfall stores necessarily more profitable, or are some stores failing to convert footfall into revenue?

In [11]:
query = """
WITH store_metrics AS (
    SELECT
        s.store_id,
        s.store_name,
        s.store_format,
        COUNT(DISTINCT f.month_id) AS months_analyzed,
        AVG(f.avg_daily_footfall) AS avg_daily_footfall,
        AVG(f.total_monthly_revenue_gbp) AS avg_monthly_revenue_gbp,
        AVG(f.net_profit_gbp) AS avg_monthly_profit_gbp,
        SUM(f.total_monthly_revenue_gbp) / NULLIF(
            SUM(
                f.avg_daily_footfall
                * EXTRACT(
                    DAY FROM (
                        DATE_TRUNC(
                            'month',
                            TO_DATE(f.month_id || '-01', 'YYYY-MM-DD')
                        ) + INTERVAL '1 month - 1 day'
                    )
                )
            ),
            0
        ) AS revenue_per_estimated_visitor_gbp
    FROM fact_store_monthly_profitability AS f
    JOIN dim_store AS s
        ON s.store_id = f.store_id
    WHERE f.month_id BETWEEN '2023-01' AND '2024-12'
    GROUP BY s.store_id, s.store_name, s.store_format
),
format_benchmarks AS (
    SELECT
        store_format,
        AVG(revenue_per_estimated_visitor_gbp) AS format_avg_revenue_per_visitor
    FROM store_metrics
    GROUP BY store_format
),
overall_stats AS (
    SELECT
        AVG(avg_daily_footfall) AS overall_avg_daily_footfall,
        CORR(avg_daily_footfall, avg_monthly_profit_gbp) AS footfall_profit_correlation,
        CORR(avg_daily_footfall, avg_monthly_revenue_gbp) AS footfall_revenue_correlation
    FROM store_metrics
)
SELECT
    m.store_id,
    m.store_name,
    m.store_format,
    m.months_analyzed,
    ROUND(m.avg_daily_footfall::numeric, 1) AS avg_daily_footfall,
    ROUND(m.avg_monthly_revenue_gbp::numeric, 2) AS avg_monthly_revenue_gbp,
    ROUND(m.avg_monthly_profit_gbp::numeric, 2) AS avg_monthly_profit_gbp,
    ROUND(m.revenue_per_estimated_visitor_gbp::numeric, 2)
        AS revenue_per_estimated_visitor_gbp,
    ROUND(b.format_avg_revenue_per_visitor::numeric, 2)
        AS format_avg_revenue_per_visitor_gbp,
    ROUND(s.footfall_profit_correlation::numeric, 3)
        AS footfall_profit_correlation,
    ROUND(s.footfall_revenue_correlation::numeric, 3)
        AS footfall_revenue_correlation,
    CASE
        WHEN m.avg_daily_footfall >= s.overall_avg_daily_footfall
         AND m.revenue_per_estimated_visitor_gbp < b.format_avg_revenue_per_visitor
        THEN 'High footfall, below-format revenue per visitor'
        ELSE 'Other'
    END AS store_assessment
FROM store_metrics AS m
JOIN format_benchmarks AS b
    ON b.store_format = m.store_format
CROSS JOIN overall_stats AS s
ORDER BY
    (m.avg_daily_footfall >= s.overall_avg_daily_footfall
     AND m.revenue_per_estimated_visitor_gbp < b.format_avg_revenue_per_visitor) DESC,
    m.avg_daily_footfall DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_id,store_name,store_format,months_analyzed,avg_daily_footfall,avg_monthly_revenue_gbp,avg_monthly_profit_gbp,revenue_per_estimated_visitor_gbp,format_avg_revenue_per_visitor_gbp,footfall_profit_correlation,footfall_revenue_correlation,store_assessment
0,39,Sunrise Coffee Central London House,High-Street,24,533.5,62415.26,8400.78,3.84,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
1,59,Sunrise Coffee Cambridge Quarter,High-Street,24,529.1,58622.05,8031.05,3.64,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
2,24,Sunrise Coffee Cheltenham Quarter,High-Street,24,525.3,67466.47,11660.18,4.21,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
3,32,Sunrise Coffee South London Corner,High-Street,24,524.1,57613.38,-2103.94,3.60,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
4,26,Sunrise Coffee Wolverhampton Yard,High-Street,24,523.9,63588.58,13032.19,3.98,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
5,13,Sunrise Coffee Norwich Station,High-Street,24,519.1,60276.24,14279.31,3.81,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
6,36,Sunrise Coffee Leicester Express,High-Street,24,517.5,63789.16,10498.34,4.05,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
7,45,Sunrise Coffee Liverpool Collective,High-Street,24,515.0,57993.27,11578.18,3.71,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
8,37,Sunrise Coffee Ipswich House,High-Street,24,509.6,66086.17,11446.88,4.25,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"
9,3,Sunrise Coffee Newport Yard,High-Street,24,505.1,65302.66,14243.94,4.25,4.25,-0.104,-0.169,"High footfall, below-format revenue per visitor"


Can we identify stores that have strong revenue but weak margins and therefore require cost optimization rather than sales growth?

In [12]:
query = """
WITH store_metrics AS (
    SELECT
        s.store_id,
        s.store_name,
        s.store_format,
        COUNT(DISTINCT f.month_id) AS months_observed,
        AVG(f.total_monthly_revenue_gbp) AS avg_monthly_revenue_gbp,
        100.0 * SUM(f.net_profit_gbp)
            / NULLIF(SUM(f.total_monthly_revenue_gbp), 0) AS profit_margin_pct,
        100.0 * SUM(f.total_monthly_cogs_gbp)
            / NULLIF(SUM(f.total_monthly_revenue_gbp), 0) AS cogs_pct_of_revenue,
        100.0 * SUM(f.monthly_rent_gbp)
            / NULLIF(SUM(f.total_monthly_revenue_gbp), 0) AS rent_pct_of_revenue,
        100.0 * SUM(f.monthly_staffing_cost_gbp)
            / NULLIF(SUM(f.total_monthly_revenue_gbp), 0) AS staffing_pct_of_revenue,
        100.0 * SUM(f.loyalty_redemption_cost_gbp)
            / NULLIF(SUM(f.total_monthly_revenue_gbp), 0) AS loyalty_cost_pct_of_revenue
    FROM fact_store_monthly_profitability AS f
    JOIN dim_store AS s
        ON s.store_id = f.store_id
    WHERE f.month_id BETWEEN '2023-01' AND '2024-12'
    GROUP BY s.store_id, s.store_name, s.store_format
    HAVING COUNT(DISTINCT f.month_id) = 24
),
format_benchmarks AS (
    SELECT
        store_format,
        PERCENTILE_CONT(0.75) WITHIN GROUP (
            ORDER BY avg_monthly_revenue_gbp
        ) AS high_revenue_threshold_gbp,
        PERCENTILE_CONT(0.25) WITHIN GROUP (
            ORDER BY profit_margin_pct
        ) AS weak_margin_threshold_pct,
        AVG(cogs_pct_of_revenue) AS format_avg_cogs_pct,
        AVG(rent_pct_of_revenue) AS format_avg_rent_pct,
        AVG(staffing_pct_of_revenue) AS format_avg_staffing_pct,
        AVG(loyalty_cost_pct_of_revenue) AS format_avg_loyalty_cost_pct
    FROM store_metrics
    GROUP BY store_format
)
SELECT
    m.store_id,
    m.store_name,
    m.store_format,
    m.months_observed,
    ROUND(m.avg_monthly_revenue_gbp::numeric, 2) AS avg_monthly_revenue_gbp,
    ROUND(m.profit_margin_pct::numeric, 2) AS profit_margin_pct,
    ROUND(b.weak_margin_threshold_pct::numeric, 2) AS format_weak_margin_threshold_pct,
    ROUND(m.cogs_pct_of_revenue::numeric, 2) AS cogs_pct_of_revenue,
    ROUND(b.format_avg_cogs_pct::numeric, 2) AS format_avg_cogs_pct,
    ROUND(m.rent_pct_of_revenue::numeric, 2) AS rent_pct_of_revenue,
    ROUND(b.format_avg_rent_pct::numeric, 2) AS format_avg_rent_pct,
    ROUND(m.staffing_pct_of_revenue::numeric, 2) AS staffing_pct_of_revenue,
    ROUND(b.format_avg_staffing_pct::numeric, 2) AS format_avg_staffing_pct,
    ROUND(m.loyalty_cost_pct_of_revenue::numeric, 2) AS loyalty_cost_pct_of_revenue,
    ROUND(b.format_avg_loyalty_cost_pct::numeric, 2) AS format_avg_loyalty_cost_pct
FROM store_metrics AS m
JOIN format_benchmarks AS b
    ON b.store_format = m.store_format
WHERE m.avg_monthly_revenue_gbp >= b.high_revenue_threshold_gbp
  AND m.profit_margin_pct <= b.weak_margin_threshold_pct
ORDER BY m.avg_monthly_revenue_gbp DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_id,store_name,store_format,months_observed,avg_monthly_revenue_gbp,profit_margin_pct,format_weak_margin_threshold_pct,cogs_pct_of_revenue,format_avg_cogs_pct,rent_pct_of_revenue,format_avg_rent_pct,staffing_pct_of_revenue,format_avg_staffing_pct,loyalty_cost_pct_of_revenue,format_avg_loyalty_cost_pct
0,46,Sunrise Coffee East London Lounge,High-Street,24,71381.52,9.61,13.79,37.74,37.89,17.17,10.42,33.92,31.25,1.57,1.94


Does higher loyalty adoption actually correlate with higher revenue or profitability?

In [14]:
query = """
SELECT 
    f.store_id,
    s.store_name,
    s.store_format,
    s.region,
    ROUND(SUM(f.loyalty_redemption_cost_gbp), 2) AS total_loyalty_cost_gbp,
    ROUND(SUM(f.total_monthly_revenue_gbp), 2) AS total_revenue_gbp,
    ROUND(SUM(f.net_profit_gbp), 2) AS total_net_profit_gbp,
    ROUND(
        (SUM(f.loyalty_redemption_cost_gbp) / NULLIF(SUM(f.total_monthly_revenue_gbp), 0)) * 100, 
        2
    ) AS loyalty_cost_to_revenue_pct,
    ROUND(
        (SUM(f.net_profit_gbp) / NULLIF(SUM(f.total_monthly_revenue_gbp), 0)) * 100, 
        2
    ) AS net_profit_margin_pct
FROM fact_store_monthly_profitability f
JOIN dim_store s ON s.store_id = f.store_id
GROUP BY f.store_id, s.store_name, s.store_format, s.region
ORDER BY total_loyalty_cost_gbp DESC;
"""

results = pd.read_sql_query(query, engine)
display(results)

,store_id,store_name,store_format,region,total_loyalty_cost_gbp,total_revenue_gbp,total_net_profit_gbp,loyalty_cost_to_revenue_pct,net_profit_margin_pct
0,59,Sunrise Coffee Cambridge Quarter,High-Street,East-of-England,33630.63,1406929.15,192745.30,2.39,13.70
1,29,Sunrise Coffee Dundee Station,High-Street,Scotland,32410.56,1434876.32,237355.12,2.26,16.54
2,28,Sunrise Coffee Hull Hub,High-Street,Yorkshire-and-Humber,32044.54,1551201.52,377641.39,2.07,24.35
3,4,Sunrise Coffee Chester Brew Bar,High-Street,North-West,32002.16,1467591.23,151239.04,2.18,10.31
4,41,Sunrise Coffee Bath Hub,High-Street,South-West,31984.80,1489695.99,173623.44,2.15,11.65
5,57,Sunrise Coffee Luton House,Transport-Hub,East-of-England,31921.68,1613269.88,418238.86,1.98,25.92
6,43,Sunrise Coffee North London Yard,High-Street,London,31402.40,1443582.05,74178.61,2.18,5.14
7,22,Sunrise Coffee Derby Lounge,High-Street,East-Midlands,31213.04,1510338.38,237267.24,2.07,15.71
8,16,Sunrise Coffee Middlesbrough Roastery,High-Street,North-East,30982.56,1527527.18,349221.97,2.03,22.86
9,35,Sunrise Coffee Canterbury Yard,High-Street,South-East,30295.86,1569786.16,294710.40,1.93,18.77


Which store format has the highest loyalty adoption rate, and does that translate into higher member revenue?

In [16]:
query = """
SELECT 
    s.store_format,
    COUNT(DISTINCT f.store_id) AS total_stores,
    ROUND(SUM(f.loyalty_redemption_cost_gbp), 2) AS total_loyalty_cost_gbp,
    ROUND(SUM(f.total_monthly_revenue_gbp), 2) AS total_revenue_gbp,
    ROUND(SUM(f.net_profit_gbp), 2) AS total_net_profit_gbp,
    ROUND(
        AVG(f.loyalty_redemption_cost_gbp / NULLIF(f.total_monthly_revenue_gbp, 0)) * 100, 
        2
    ) AS avg_loyalty_cost_to_revenue_pct,
    ROUND(
        AVG(f.net_profit_margin_pct) * 100, 
        2
    ) AS avg_net_profit_margin_pct
FROM fact_store_monthly_profitability f
JOIN dim_store s ON s.store_id = f.store_id
GROUP BY s.store_format
ORDER BY avg_loyalty_cost_to_revenue_pct DESC;
"""

results = pd.read_sql_query(query, engine)
display(results)

,store_format,total_stores,total_loyalty_cost_gbp,total_revenue_gbp,total_net_profit_gbp,avg_loyalty_cost_to_revenue_pct,avg_net_profit_margin_pct
0,High-Street,58,1669804.36,86499463.27,16078392.61,2.13,14.34
1,Transport-Hub,2,60356.86,3186525.91,801111.46,1.98,22.53


Are there stores with high loyalty adoption but low average transaction value? What could this indicate about customer behaviour?

In [19]:
query = """
WITH stores AS (
    SELECT
        s.store_id,
        s.store_name,
        s.store_format,
        s.loyalty_adoption_rate,
        SUM(d.total_revenue_gbp) / NULLIF(SUM(d.total_transactions), 0)
            AS avg_transaction_value_gbp,
        100.0 * SUM(d.loyalty_transactions)
            / NULLIF(SUM(d.total_transactions), 0) AS loyalty_transaction_pct
    FROM dim_store AS s
    JOIN fact_daily_sales AS d USING (store_id)
    GROUP BY s.store_id, s.store_name, s.store_format, s.loyalty_adoption_rate
),
benchmarks AS (
    SELECT
        store_format,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY loyalty_adoption_rate)
            AS high_adoption_threshold,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY avg_transaction_value_gbp)
            AS low_transaction_threshold
    FROM stores
    GROUP BY store_format
)
SELECT
    s.store_id,
    s.store_name,
    s.store_format,
    ROUND((s.loyalty_adoption_rate * 100)::numeric, 2) AS loyalty_adoption_pct,
    ROUND(s.avg_transaction_value_gbp::numeric, 2) AS avg_transaction_value_gbp,
    ROUND(s.loyalty_transaction_pct::numeric, 2) AS loyalty_transaction_pct
FROM stores AS s
JOIN benchmarks AS b USING (store_format)
WHERE s.loyalty_adoption_rate >= b.high_adoption_threshold
  AND s.avg_transaction_value_gbp <= b.low_transaction_threshold
ORDER BY s.loyalty_adoption_rate DESC, s.avg_transaction_value_gbp;

"""

result = pd.read_sql_query(query, engine)
display(result)

,store_id,store_name,store_format,loyalty_adoption_pct,avg_transaction_value_gbp,loyalty_transaction_pct
0,50,Sunrise Coffee Bradford Brew Bar,High-Street,57.3,7.08,53.82
1,11,Sunrise Coffee Cambridge Brew Bar,High-Street,54.6,7.19,52.37
2,57,Sunrise Coffee Luton House,Transport-Hub,51.8,7.37,49.03
3,44,Sunrise Coffee Newcastle Collective,High-Street,48.0,7.15,45.14


What percentage of total revenue comes from espresso, filter coffee, food, and retail beans?

In [20]:
query = """
WITH totals AS (
    SELECT
        SUM(total_revenue_gbp) AS total_revenue,
        SUM(espresso_revenue_gbp) AS espresso,
        SUM(filter_revenue_gbp) AS filter_coffee,
        SUM(food_revenue_gbp) AS food,
        SUM(retail_beans_revenue_gbp) AS retail_beans
    FROM fact_daily_sales
)
SELECT
    category,
    ROUND(revenue::numeric, 2) AS revenue_gbp,
    ROUND((100.0 * revenue / NULLIF(total_revenue, 0))::numeric, 2)
        AS pct_of_total_revenue
FROM totals
CROSS JOIN LATERAL (
    VALUES
        ('Espresso', espresso),
        ('Filter coffee', filter_coffee),
        ('Food', food),
        ('Retail beans', retail_beans)
) AS categories(category, revenue)
ORDER BY revenue DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,category,revenue_gbp,pct_of_total_revenue
0,Espresso,9346953.46,45.70
1,Food,5332403.73,26.07
2,Filter coffee,3924072.32,19.19
3,Retail beans,1850210.70,9.05


Which product/revenue category contributes the most to revenue, and which contributes the most to gross profit?

In [21]:
query = """
WITH categories AS (
    SELECT
        p.product_category,
        SUM(f.total_revenue_gbp) AS revenue_gbp,
        SUM(f.daily_gross_profit_gbp) AS gross_profit_gbp
    FROM fact_daily_sales AS f
    JOIN dim_product AS p USING (product_id)
    GROUP BY p.product_category
)
SELECT
    product_category,
    ROUND(revenue_gbp::numeric, 2) AS revenue_gbp,
    ROUND(100 * revenue_gbp / SUM(revenue_gbp) OVER (), 2) AS revenue_share_pct,
    ROUND(gross_profit_gbp::numeric, 2) AS gross_profit_gbp,
    ROUND(100 * gross_profit_gbp / NULLIF(SUM(gross_profit_gbp) OVER (), 0), 2)
        AS gross_profit_share_pct,
    RANK() OVER (ORDER BY revenue_gbp DESC) AS revenue_rank,
    RANK() OVER (ORDER BY gross_profit_gbp DESC) AS gross_profit_rank
FROM categories
ORDER BY revenue_rank;
"""

result = pd.read_sql_query(query, engine)
display(result)

,product_category,revenue_gbp,revenue_share_pct,gross_profit_gbp,gross_profit_share_pct,revenue_rank,gross_profit_rank
0,Espresso-Drinks,6268185.25,30.65,3948848.97,30.66,1,1
1,Food,4990312.29,24.40,3138948.79,24.37,2,2
2,Retail-Beans,4137962.59,20.23,2598834.36,20.18,3,3
3,Cold-Drinks,2042188.35,9.98,1287343.90,10.00,4,4
4,Filter-Coffee,2021515.16,9.88,1282652.26,9.96,5,5
5,Merchandise,993476.57,4.86,622805.42,4.84,6,6


Which stores have an unusually high or low food revenue share compared with the chain average?

In [22]:
query = """
WITH store_shares AS (
    SELECT
        s.store_id,
        s.store_name,
        s.region,
        100.0 * SUM(f.food_revenue_gbp)
            / NULLIF(SUM(f.total_revenue_gbp), 0) AS food_share_pct
    FROM fact_daily_sales AS f
    JOIN dim_store AS s USING (store_id)
    GROUP BY s.store_id, s.store_name, s.region
),
benchmarks AS (
    SELECT
        100.0 * SUM(food_revenue_gbp)
            / NULLIF(SUM(total_revenue_gbp), 0) AS chain_food_share_pct
    FROM fact_daily_sales
),
spread AS (
    SELECT STDDEV_POP(food_share_pct) AS store_share_stddev
    FROM store_shares
)
SELECT
    s.store_id,
    s.store_name,
    s.region,
    ROUND(s.food_share_pct::numeric, 2) AS food_share_pct,
    ROUND(b.chain_food_share_pct::numeric, 2) AS chain_avg_pct,
    ROUND((s.food_share_pct - b.chain_food_share_pct)::numeric, 2)
        AS difference_pp,
    CASE
        WHEN s.food_share_pct > b.chain_food_share_pct + d.store_share_stddev
            THEN 'Unusually high'
        WHEN s.food_share_pct < b.chain_food_share_pct - d.store_share_stddev
            THEN 'Unusually low'
        ELSE 'Within typical range'
    END AS comparison
FROM store_shares AS s
CROSS JOIN benchmarks AS b
CROSS JOIN spread AS d
ORDER BY difference_pp DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_id,store_name,region,food_share_pct,chain_avg_pct,difference_pp,comparison
0,47,Sunrise Coffee Exeter Express,South-West,28.53,26.07,2.45,Unusually high
1,46,Sunrise Coffee East London Lounge,London,28.10,26.07,2.03,Unusually high
2,55,Sunrise Coffee Swansea Collective,Wales,28.02,26.07,1.94,Unusually high
3,15,Sunrise Coffee West London Quarter,London,27.99,26.07,1.92,Unusually high
4,48,Sunrise Coffee Reading Hub,South-East,27.89,26.07,1.82,Unusually high
5,26,Sunrise Coffee Wolverhampton Yard,West-Midlands,27.66,26.07,1.59,Unusually high
6,20,Sunrise Coffee Guildford Hub,South-East,27.56,26.07,1.49,Unusually high
7,41,Sunrise Coffee Bath Hub,South-West,27.48,26.07,1.41,Unusually high
8,37,Sunrise Coffee Ipswich House,East-of-England,27.45,26.07,1.38,Unusually high
9,1,Sunrise Coffee Sunderland Station,North-East,27.34,26.07,1.27,Unusually high


Are stores overly dependent on one particular revenue category?

In [23]:
query = """
WITH category_revenue AS (
    SELECT
        s.store_id,
        s.store_name,
        p.product_category,
        SUM(f.total_revenue_gbp) AS revenue
    FROM fact_daily_sales AS f
    JOIN dim_store AS s USING (store_id)
    JOIN dim_product AS p USING (product_id)
    GROUP BY s.store_id, s.store_name, p.product_category
),
shares AS (
    SELECT
        *,
        revenue / NULLIF(SUM(revenue) OVER (PARTITION BY store_id), 0) AS share
    FROM category_revenue
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY share DESC) AS category_rank,
        SUM(share * share) OVER (PARTITION BY store_id) AS hhi
    FROM shares
)
SELECT
    store_id,
    store_name,
    product_category AS largest_revenue_category,
    ROUND((share * 100)::numeric, 2) AS largest_category_share_pct,
    ROUND((hhi * 10000)::numeric, 0) AS revenue_concentration_hhi,
    CASE
        WHEN share >= 0.60 THEN 'Highly dependent'
        WHEN share >= 0.40 THEN 'Moderately dependent'
        ELSE 'More diversified'
    END AS assessment
FROM ranked
WHERE category_rank = 1
ORDER BY largest_category_share_pct DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_id,store_name,largest_revenue_category,largest_category_share_pct,revenue_concentration_hhi,assessment
0,18,Sunrise Coffee Nottingham Quarter,Espresso-Drinks,36.57,2380.0,More diversified
1,2,Sunrise Coffee Bradford Yard,Espresso-Drinks,35.88,2282.0,More diversified
2,56,Sunrise Coffee Lincoln Corner,Espresso-Drinks,35.87,2186.0,More diversified
3,34,Sunrise Coffee Brighton Express,Espresso-Drinks,35.77,2404.0,More diversified
4,31,Sunrise Coffee Southampton House,Espresso-Drinks,35.66,2318.0,More diversified
5,52,Sunrise Coffee Chester House,Espresso-Drinks,35.65,2387.0,More diversified
6,24,Sunrise Coffee Cheltenham Quarter,Espresso-Drinks,35.63,2429.0,More diversified
7,45,Sunrise Coffee Liverpool Collective,Espresso-Drinks,35.62,2287.0,More diversified
8,54,Sunrise Coffee Derry Roastery,Espresso-Drinks,34.97,2320.0,More diversified
9,12,Sunrise Coffee Cardiff Corner,Espresso-Drinks,34.77,2234.0,More diversified


Which days of the week generate the highest revenue and transaction volume?

In [24]:
query = """
SELECT
    d.day_of_week,
    COUNT(DISTINCT d.date_id) AS days_observed,
    ROUND(SUM(f.total_revenue_gbp)::numeric, 2) AS total_revenue_gbp,
    ROUND(
        (SUM(f.total_revenue_gbp) / COUNT(DISTINCT d.date_id))::numeric, 2
    ) AS avg_daily_revenue_gbp,
    SUM(f.total_transactions) AS total_transactions,
    ROUND(
        (SUM(f.total_transactions)::numeric / COUNT(DISTINCT d.date_id)), 0
    ) AS avg_daily_transactions
FROM fact_daily_sales AS f
JOIN dim_date AS d USING (date_id)
GROUP BY d.day_of_week
ORDER BY avg_daily_revenue_gbp DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,day_of_week,days_observed,total_revenue_gbp,avg_daily_revenue_gbp,total_transactions,avg_daily_transactions
0,Thursday,104,3294633.29,31679.17,453499,4361.0
1,Friday,104,3166107.27,30443.34,433437,4168.0
2,Monday,105,3171625.19,30205.95,434330,4136.0
3,Wednesday,104,3096443.80,29773.50,425321,4090.0
4,Tuesday,104,3054984.82,29374.85,420272,4041.0
5,Saturday,104,2359478.74,22687.30,325290,3128.0
6,Sunday,105,2310367.10,22003.50,316906,3018.0


How different is weekend performance from weekday performance across store formats?

In [25]:
query = """
WITH store_days AS (
    SELECT
        s.store_id,
        s.store_format,
        d.date_id,
        d.is_weekend,
        SUM(f.total_revenue_gbp) AS revenue,
        SUM(f.total_transactions) AS transactions,
        SUM(f.daily_gross_profit_gbp) AS gross_profit
    FROM fact_daily_sales AS f
    JOIN dim_store AS s USING (store_id)
    JOIN dim_date AS d USING (date_id)
    GROUP BY s.store_id, s.store_format, d.date_id, d.is_weekend
)
SELECT
    store_format,
    ROUND(AVG(revenue) FILTER (WHERE is_weekend)::numeric, 2) AS weekend_revenue,
    ROUND(AVG(revenue) FILTER (WHERE NOT is_weekend)::numeric, 2) AS weekday_revenue,
    ROUND((100 * (
        AVG(revenue) FILTER (WHERE is_weekend)
        / NULLIF(AVG(revenue) FILTER (WHERE NOT is_weekend), 0) - 1
    ))::numeric, 2) AS revenue_change_pct,
    ROUND(AVG(transactions) FILTER (WHERE is_weekend)::numeric, 1) AS weekend_transactions,
    ROUND(AVG(transactions) FILTER (WHERE NOT is_weekend)::numeric, 1) AS weekday_transactions,
    ROUND((100 * (
        AVG(transactions) FILTER (WHERE is_weekend)
        / NULLIF(AVG(transactions) FILTER (WHERE NOT is_weekend), 0) - 1
    ))::numeric, 2) AS transaction_change_pct,
    ROUND(AVG(gross_profit) FILTER (WHERE is_weekend)::numeric, 2) AS weekend_gross_profit,
    ROUND(AVG(gross_profit) FILTER (WHERE NOT is_weekend)::numeric, 2) AS weekday_gross_profit
FROM store_days
GROUP BY store_format
ORDER BY store_format;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_format,weekend_revenue,weekday_revenue,revenue_change_pct,weekend_transactions,weekday_transactions,transaction_change_pct,weekend_gross_profit,weekday_gross_profit
0,High-Street,1845.42,2449.53,-24.66,254.0,336.5,-24.54,1162.11,1542.28
1,Transport-Hub,2095.88,3086.31,-32.09,282.1,416.3,-32.25,1328.35,1944.72


Do UK bank holidays increase revenue and footfall, or do some store formats perform worse on bank holidays?

In [26]:
query = """
WITH store_days AS (
    SELECT
        s.store_id,
        s.store_format,
        d.date_id,
        d.is_uk_bank_holiday,
        SUM(f.total_revenue_gbp) AS revenue,
        SUM(f.total_transactions) AS transactions,
        MAX(f.footfall) AS footfall
    FROM fact_daily_sales AS f
    JOIN dim_store AS s USING (store_id)
    JOIN dim_date AS d USING (date_id)
    GROUP BY s.store_id, s.store_format, d.date_id, d.is_uk_bank_holiday
)
SELECT
    store_format,
    COUNT(*) FILTER (WHERE is_uk_bank_holiday) AS bank_holiday_store_days,
    ROUND(AVG(revenue) FILTER (WHERE is_uk_bank_holiday)::numeric, 2) AS holiday_revenue,
    ROUND(AVG(revenue) FILTER (WHERE NOT is_uk_bank_holiday)::numeric, 2) AS regular_revenue,
    ROUND((100 * (
        AVG(revenue) FILTER (WHERE is_uk_bank_holiday)
        / NULLIF(AVG(revenue) FILTER (WHERE NOT is_uk_bank_holiday), 0) - 1
    ))::numeric, 2) AS revenue_change_pct,
    ROUND(AVG(footfall) FILTER (WHERE is_uk_bank_holiday)::numeric, 1) AS holiday_footfall,
    ROUND(AVG(footfall) FILTER (WHERE NOT is_uk_bank_holiday)::numeric, 1) AS regular_footfall,
    ROUND((100 * (
        AVG(footfall) FILTER (WHERE is_uk_bank_holiday)
        / NULLIF(AVG(footfall) FILTER (WHERE NOT is_uk_bank_holiday), 0) - 1
    ))::numeric, 2) AS footfall_change_pct
FROM store_days
GROUP BY store_format
ORDER BY revenue_change_pct DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_format,bank_holiday_store_days,holiday_revenue,regular_revenue,revenue_change_pct,holiday_footfall,regular_footfall,footfall_change_pct
0,High-Street,210,1540.94,2296.34,-32.90,252.1,381.3,-33.89
1,Transport-Hub,8,1793.90,2869.44,-37.48,306.6,467.9,-34.47


Does average dwell time have any meaningful relationship with average transaction value or revenue?

In [ ]:
query = """
SELECT 
    f.store_id,
    s.store_name,
    s.store_format,
    s.region,
    ROUND(AVG(d.avg_dwell_time_minutes), 2) AS avg_dwell_time_minutes,
    ROUND(AVG(d.avg_transaction_value_gbp), 2) AS avg_transaction_value_gbp,
    ROUND(SUM(f.total_monthly_revenue_gbp), 2) AS total_revenue_gbp,
    ROUND(SUM(f.net_profit_gbp), 2) AS total_net_profit_gbp
FROM fact_store_monthly_profitability f
JOIN dim_store s ON s.store_id = f.store_id
LEFT JOIN (
    SELECT 
        store_id, 
        ROUND(AVG(avg_dwell_time_minutes), 2) AS avg_dwell_time_minutes,
        ROUND(AVG(sales_amount_gbp / NULLIF(transaction_count, 0)), 2) AS avg_transaction_value_gbp
    FROM fact_daily_sales
    GROUP BY store_id
) d ON f.store_id = d.store_id
GROUP BY f.store_id, s.store_name, s.store_format, s.region
ORDER BY avg_dwell_time_minutes DESC;
"""

results = pd.read_sql_query(query, engine)
display(results)

,store_days,dwell_vs_transaction_value_corr,dwell_vs_revenue_corr
0,8910,-0.011,0.12


Are stores with higher staffing levels actually generating enough additional revenue to justify the labour cost?

In [3]:
query = """
WITH daily AS (
    SELECT
        store_id,
        date_id,
        MAX(staff_count) AS staff_count,
        MAX(total_staff_hours) AS staff_hours,
        SUM(total_revenue_gbp) AS revenue
    FROM fact_daily_sales
    GROUP BY store_id, date_id
),
store_daily AS (
    SELECT
        store_id,
        AVG(staff_count) AS avg_daily_staff,
        AVG(revenue) AS avg_daily_revenue,
        SUM(revenue) / NULLIF(SUM(staff_hours), 0) AS revenue_per_staff_hour
    FROM daily
    GROUP BY store_id
),
store_monthly AS (
    SELECT
        store_id,
        AVG(total_monthly_revenue_gbp) AS avg_monthly_revenue,
        AVG(monthly_staffing_cost_gbp) AS avg_monthly_labour_cost,
        AVG(net_profit_gbp) AS avg_monthly_profit
    FROM fact_store_monthly_profitability
    GROUP BY store_id
)
SELECT
    s.store_name,
    s.store_format,
    ROUND(d.avg_daily_staff::numeric, 1) AS avg_daily_staff,
    ROUND(d.avg_daily_revenue::numeric, 2) AS avg_daily_revenue_gbp,
    ROUND(d.revenue_per_staff_hour::numeric, 2) AS revenue_per_staff_hour_gbp,
    ROUND(m.avg_monthly_labour_cost::numeric, 2) AS avg_monthly_labour_cost_gbp,
    ROUND(
        (100 * m.avg_monthly_labour_cost / NULLIF(m.avg_monthly_revenue, 0))::numeric,
        2
    ) AS labour_cost_pct_of_revenue,
    ROUND(m.avg_monthly_profit::numeric, 2) AS avg_monthly_profit_gbp
FROM store_daily AS d
JOIN store_monthly AS m USING (store_id)
JOIN dim_store AS s USING (store_id)
ORDER BY labour_cost_pct_of_revenue DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_name,store_format,avg_daily_staff,avg_daily_revenue_gbp,revenue_per_staff_hour_gbp,avg_monthly_labour_cost_gbp,labour_cost_pct_of_revenue,avg_monthly_profit_gbp
0,Sunrise Coffee South London Corner,High-Street,5.4,2214.79,55.65,29781.38,51.69,-2103.94
1,Sunrise Coffee Manchester Collective,High-Street,5.7,2256.37,53.25,26275.23,46.51,2773.01
2,Sunrise Coffee Nottingham Quarter,High-Street,5.6,2273.27,54.71,24920.38,42.69,5502.57
3,Sunrise Coffee Chester Brew Bar,High-Street,5.6,2313.73,55.24,24916.00,40.75,6301.63
4,Sunrise Coffee Newcastle Collective,High-Street,5.8,2203.11,50.24,24361.35,40.43,8469.74
5,Sunrise Coffee Sunderland Express,High-Street,5.5,2205.29,51.00,23327.09,40.32,7510.92
6,Sunrise Coffee Edinburgh Corner,High-Street,5.9,2284.25,51.72,25199.92,39.45,9781.63
7,Sunrise Coffee Leeds Brew Bar,High-Street,5.5,2196.18,52.15,24516.64,39.37,7770.57
8,Sunrise Coffee Birmingham Roastery,High-Street,5.7,2224.73,52.68,25019.72,38.24,9986.97
9,Sunrise Coffee Cambridge Quarter,High-Street,5.3,2242.89,57.03,21583.35,36.82,8031.05


Which stores show unusually volatile monthly profitability compared with the rest of the chain?

In [5]:
query = """
SELECT 
    f.store_id,
    s.store_name,
    s.store_format,
    s.region,
    COUNT(*) AS total_months,
    ROUND(AVG(f.net_profit_gbp), 2) AS avg_monthly_net_profit_gbp,
    ROUND(STDDEV(f.net_profit_gbp), 2) AS stddev_net_profit_gbp,
    ROUND(
        (STDDEV(f.net_profit_gbp) / NULLIF(ABS(AVG(f.net_profit_gbp)), 0)) * 100, 
        2
    ) AS profit_volatility_cv_pct,
    ROUND(MIN(f.net_profit_gbp), 2) AS min_monthly_net_profit_gbp,
    ROUND(MAX(f.net_profit_gbp), 2) AS max_monthly_net_profit_gbp,
    ROUND(MAX(f.net_profit_gbp) - MIN(f.net_profit_gbp), 2) AS profit_range_gbp
FROM fact_store_monthly_profitability f
JOIN dim_store s ON s.store_id = f.store_id
GROUP BY f.store_id, s.store_name, s.store_format, s.region
HAVING COUNT(*) > 1
ORDER BY profit_volatility_cv_pct DESC
LIMIT 10;
"""

results = pd.read_sql_query(query, engine)
display(results)

,store_id,store_name,store_format,region,total_months,avg_monthly_net_profit_gbp,stddev_net_profit_gbp,profit_volatility_cv_pct,min_monthly_net_profit_gbp,max_monthly_net_profit_gbp,profit_range_gbp
0,53,Sunrise Coffee Canary Wharf Express,High-Street,London,24,1783.64,10869.74,609.41,-16891.42,20557.46,37448.88
1,32,Sunrise Coffee South London Corner,High-Street,London,24,-2103.94,11348.96,539.41,-23798.79,16116.44,39915.23
2,19,Sunrise Coffee Manchester Collective,High-Street,North-West,24,2773.01,11878.15,428.35,-24382.42,19368.95,43751.37
3,43,Sunrise Coffee North London Yard,High-Street,London,24,3090.78,6912.06,223.64,-8539.89,19267.85,27807.74
4,18,Sunrise Coffee Nottingham Quarter,High-Street,East-Midlands,24,5502.57,10047.63,182.60,-15248.79,26199.25,41448.04
5,47,Sunrise Coffee Exeter Express,High-Street,South-West,24,6183.24,11025.95,178.32,-8588.71,39619.41,48208.12
6,4,Sunrise Coffee Chester Brew Bar,High-Street,North-West,24,6301.63,11212.69,177.93,-21445.34,26758.08,48203.42
7,5,Sunrise Coffee Canary Wharf Quarter,High-Street,London,24,5475.94,9305.13,169.93,-10381.25,27175.20,37556.45
8,46,Sunrise Coffee East London Lounge,High-Street,London,24,6860.17,11479.91,167.34,-15444.11,25883.75,41327.86
9,49,Sunrise Coffee Sunderland Express,High-Street,North-East,24,7510.92,10727.85,142.83,-13588.23,27689.53,41277.76


Are there specific periods where revenue increases but profitability decreases? What explains the divergence?

In [13]:
query = """
WITH monthly AS (
    SELECT month_id,
           SUM(total_monthly_revenue_gbp) AS revenue,
           SUM(net_profit_gbp) AS profit,
           SUM(total_monthly_cogs_gbp) AS cogs,
           SUM(monthly_staffing_cost_gbp) AS staffing
    FROM fact_store_monthly_profitability
    GROUP BY month_id
),
changes AS (
    SELECT *,
           revenue - LAG(revenue) OVER (ORDER BY month_id) AS revenue_change,
           profit - LAG(profit) OVER (ORDER BY month_id) AS profit_change,
           cogs - LAG(cogs) OVER (ORDER BY month_id) AS cogs_change,
           staffing - LAG(staffing) OVER (ORDER BY month_id) AS staffing_change
    FROM monthly
)
SELECT month_id,
       ROUND(revenue_change::numeric, 2) AS revenue_change_gbp,
       ROUND(profit_change::numeric, 2) AS profit_change_gbp,
       ROUND(cogs_change::numeric, 2) AS cogs_change_gbp,
       ROUND(staffing_change::numeric, 2) AS staffing_change_gbp,
       CASE
           WHEN revenue_change IS NULL OR profit_change IS NULL THEN 'No prior month'
           WHEN revenue_change > 0 AND profit_change < 0 THEN 'Revenue up, profit down'
           WHEN revenue_change < 0 AND profit_change > 0 THEN 'Revenue down, profit up'
           ELSE 'Same direction or unchanged'
       END AS result
FROM changes
ORDER BY month_id;
"""

result = pd.read_sql_query(query, engine)
display(result)

,month_id,revenue_change_gbp,profit_change_gbp,cogs_change_gbp,staffing_change_gbp,result
0,2023-01,NaN,NaN,NaN,NaN,No prior month
1,2023-02,-38977.71,-28221.75,25410.58,-39646.61,Same direction or unchanged
2,2023-03,99915.19,63355.48,9850.16,28301.50,Same direction or unchanged
3,2023-04,-47865.37,18388.69,-18235.57,-45233.41,"Revenue down, profit up"
4,2023-05,-60999.20,-72351.21,3362.07,4754.34,Same direction or unchanged
5,2023-06,7174.92,11935.43,-40245.58,34572.23,Same direction or unchanged
6,2023-07,149487.90,106764.54,75117.86,-29450.10,Same direction or unchanged
7,2023-08,-111309.91,-67981.47,-62284.72,16627.76,Same direction or unchanged
8,2023-09,-73219.18,-78785.65,3595.91,-1652.82,Same direction or unchanged
9,2023-10,29702.59,26072.77,-2327.07,4069.00,Same direction or unchanged


What are the strongest and weakest trading periods for each store format?

In [14]:
query = """
WITH periods AS (
    SELECT
        s.store_format,
        f.month_id,
        COUNT(DISTINCT f.store_id) AS stores,
        AVG(f.total_monthly_revenue_gbp) AS avg_revenue,
        AVG(f.net_profit_gbp) AS avg_profit,
        100.0 * SUM(f.net_profit_gbp)
            / NULLIF(SUM(f.total_monthly_revenue_gbp), 0) AS margin_pct
    FROM fact_store_monthly_profitability AS f
    JOIN dim_store AS s USING (store_id)
    GROUP BY s.store_format, f.month_id
),
ranked AS (
    SELECT *,
        RANK() OVER (PARTITION BY store_format ORDER BY avg_profit DESC) AS strongest,
        RANK() OVER (PARTITION BY store_format ORDER BY avg_profit) AS weakest
    FROM periods
)
SELECT
    store_format,
    month_id,
    CASE WHEN strongest = 1 THEN 'Strongest' ELSE 'Weakest' END AS period,
    stores,
    ROUND(avg_revenue::numeric, 2) AS avg_revenue_gbp,
    ROUND(avg_profit::numeric, 2) AS avg_profit_gbp,
    ROUND(margin_pct::numeric, 2) AS profit_margin_pct
FROM ranked
WHERE strongest = 1 OR weakest = 1
ORDER BY store_format, avg_profit DESC;
"""

result = pd.read_sql_query(query, engine)
display(result)

,store_format,month_id,period,stores,avg_revenue_gbp,avg_profit_gbp,profit_margin_pct
0,High-Street,2024-11,Strongest,58,64838.90,13506.69,20.83
1,High-Street,2023-11,Weakest,58,59348.57,9235.16,15.56
2,Transport-Hub,2024-06,Strongest,2,87437.95,29431.53,33.66
3,Transport-Hub,2024-01,Weakest,2,52142.00,8060.48,15.46


In [5]:

query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT store_id) AS total_stores,
    COUNT(DISTINCT date_id) AS total_days,
    COUNT(DISTINCT store_id::text || '-' || date_id::text) AS store_day_combinations,
    COALESCE(SUM(total_revenue_gbp), 0) AS daily_revenue
FROM public.fact_daily_sales;
"""

result = pd.read_sql_query(query, engine)
display(result)

,total_rows,total_stores,total_days,store_day_combinations,daily_revenue
0,10000,60,730,8910,20453640.21


In [7]:
query = """
WITH daily_store_month_revenue AS (
    SELECT
        f.store_id,
        d.month_id,
        SUM(f.total_revenue_gbp) AS daily_revenue
    FROM public.fact_daily_sales f
    JOIN public.dim_date d
        ON f.date_id = d.date_id
    WHERE d.full_date >= DATE '2023-01-01'
      AND d.full_date < DATE '2025-01-01'
    GROUP BY f.store_id, d.month_id
),
monthly_store_revenue AS (
    SELECT
        store_id,
        month_id,
        SUM(total_monthly_revenue_gbp) AS monthly_revenue
    FROM public.fact_store_monthly_profitability
    GROUP BY store_id, month_id
)
SELECT
    COALESCE(SUM(d.daily_revenue), 0) AS total_daily_revenue,
    COALESCE(SUM(m.monthly_revenue), 0) AS total_monthly_revenue,
    COALESCE(SUM(m.monthly_revenue), 0) - COALESCE(SUM(d.daily_revenue), 0) AS revenue_difference
FROM daily_store_month_revenue d
FULL OUTER JOIN monthly_store_revenue m
    ON d.store_id = m.store_id
   AND d.month_id = m.month_id;
"""

result = pd.read_sql_query(query, engine)
display(result)

,total_daily_revenue,total_monthly_revenue,revenue_difference
0,20453640.21,89685989.18,69232348.97
